In [1]:
!pip install torch -q
!pip install transformers -q
!pip install numpy -q
!pip install langchain -q
!pip install langchain_community -q
!pip install langchain-chroma -q
!pip install sentence_transformers -q
!pip install -U langchain-huggingface -q
!pip install -U langchain-huggingface huggingface_hub

In [2]:
!pip install -U langchain langchain-core langchain-huggingface huggingface_hub

In [3]:
## Initialize HuggingFace LLM

In [27]:
from langchain_huggingface import HuggingFaceEndpoint
import os

from google.colab import userdata
hf_token = userdata.get("HF_TOKEN")

llm = HuggingFaceEndpoint(
    repo_id  = "mistralai/Mistral-7B-v0.1",
    temperature=0.5,
    max_new_tokens=500,
    huggingfacehub_api_token = "HF_TOKEN"
)

In [28]:
## Initialize Embedding Model

In [29]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-mpnet-base-v2"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [30]:
## Initialize Output Parser

In [31]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()


In [32]:
## Load PDF Document

In [33]:
!pip install pypdf -qU

In [34]:
from langchain_community.document_loaders import PyPDFLoader

pdf_folder = "/content/Content"

docs = []
for filename in sorted(os.listdir(pdf_folder)):
    if filename.lower().endswith(".pdf"):
        filepath = os.path.join(pdf_folder, filename)
        loader = PyPDFLoader(filepath)
        paper_docs = loader.load()
        paper_title = os.path.splitext(filename)[0]
        for doc in paper_docs:
            doc.metadata["paper_title"] = paper_title
        docs.extend(paper_docs)
        print(f"Loaded {len(paper_docs)} pages from {filename}")

Loaded 33 pages from 1-s2.0-S1566253526003246-main.pdf
Loaded 15 pages from Glob Bus Org Exc - 2025 - Islam - The Rise of Agentic AI  Synthesis of Current Knowledge and Future Research Agenda.pdf
Loaded 34 pages from Toward_Edge_General_Intelligence_With_Agentic_AI_and_Agentification_Concepts_Technologies_and_Future_Directions.pdf


In [35]:
len(docs)

82

In [36]:
## Split documents into Chunks

In [37]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)
print(len(splits))

628


In [38]:
## Create Vector Store and Retriever

In [39]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(documents=splits, embedding=embedding_model)

In [40]:
retriever = vectorstore.as_retriever()

In [41]:
## Define Prompt Template

In [42]:
from langchain_core.prompts import ChatPromptTemplate

template = """
Answer this question using the provided context only. When you use information
from a specific source, reference it using its [n] number from the context.

{question}

Context:
{context}

Answer:
"""

prompt = ChatPromptTemplate.from_template(template)

In [43]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\nAnswer this question using the provided context only. When you use information\nfrom a specific source, reference it using its [n] number from the context.\n\n{question}\n\nContext:\n{context}\n\nAnswer:\n'), additional_kwargs={})])

In [44]:
## Chain Retriever and Prompt Template with LLM (citation-grounded)

In [45]:
def format_docs_with_sources(docs):
    formatted = []
    for i, doc in enumerate(docs):
        paper = doc.metadata.get("paper_title", "unknown paper")
        page = doc.metadata.get("page", "unknown")
        formatted.append(f"[{i+1}] (Paper: {paper}, page {page})\n{doc.page_content}")
    return "\n\n".join(formatted)

In [46]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

rag_chain_with_sources = RunnableParallel(
    {"context": retriever, "question": RunnablePassthrough()}
).assign(
    answer=(
        {
            "context": lambda x: format_docs_with_sources(x["context"]),
            "question": lambda x: x["question"],
        }
        | prompt
        | llm
        | output_parser
    )
)

def ask(question):
    result = rag_chain_with_sources.invoke(question)
    print("Answer:\n")
    print(result["answer"])
    print("\nSources:")
    seen = set()
    for doc in result["context"]:
        key = (doc.metadata.get("paper_title", "unknown paper"), doc.metadata.get("page", "?"))
        if key not in seen:
            seen.add(key)
            print(f"- {key[0]}, page {key[1]}")
    return result

In [47]:
## Invoke RAG Chain with Example Questions

In [48]:
_ = ask("What is Agentic AI?")

Answer:


Agentic AI is a broad paradigm and design philosophy for individual agents rather than a specific instantiation. It refers to the use of language models (LLMs) as the reasoning core of an agent, which is then deployed as an agentic AI system. Agentic AI systems are autonomous agents that operate with minimal or no human supervision and have a core operational cycle of perception, reasoning, and acting in the environment. They use a planner/controller to translate high-level goals into executable sub-tasks or plans, and an executor/tool caller to invoke APIs, code, or tools to perform concrete actions. They also use reﬂexion to self-evaluate and improve via feedback on past actions.

Sources:
- 1-s2.0-S1566253526003246-main, page 5
- Glob Bus Org Exc - 2025 - Islam - The Rise of Agentic AI  Synthesis of Current Knowledge and Future Research Agenda, page 14


In [49]:
_ = ask("What are the latest findings in Agentic AI?")

Answer:


In the field of agentic AI, there have been a number of recent advancements and findings. One of the most significant developments has been the emergence of large language models (LLMs), such as ChatGPT and GPT-4, which have shown remarkable progress in natural language processing (NLP) tasks. These models have been trained on vast amounts of data and have demonstrated impressive performance in tasks such as language translation, summarization, and question answering.

Another significant finding in agentic AI is the development of autonomous agents that can make decisions and take actions without human intervention. These agents are designed to operate in complex environments and can learn from their experiences to improve their performance over time. One example of an autonomous agent is the AlphaGo Zero system, which was developed by DeepMind and was able to defeat the world champion in the game of Go without any prior knowledge of the game.

Another finding in agentic AI 

In [50]:
_ = ask("What are the future findings expected in Agentic AI?")

Answer:


[1] (Paper: 1-s2.0-S1566253526003246-main, page 1)
A. Farooq et al.
Fig. 1. Illustrative timeline of representative agentic AI systems proposed between 2023 and 2025, highlighting major architectural paradigms and emerging research 
directions.
Fig. 2. Taxonomy of Benchmarks and Evaluation Metrics for Agentic Lifecycle. Major literature references for each lifecycle stage of agentic AI. Perception [7,16,
22,65,66,68,73,83,87,92,94–96,99,125,132,136,145,156,167,207,208] Planning [2,8,24,32,33,45,48,68,70,71,74,75,91,94,97,98,117–120,131,133,146,155,166].
Tool [12,17,72,78,79,88–90,93,115,123–125,133,139,146–148,152,169,205,206] Memory [10,11,13,18,21,34,40,46,54,64,69,76,77,85,94,123–125] Action [19,
23,25,35,47,54,66,68,80,81,84,86,94,115,119,121,124,125,138]
Information Fusion 136 (2026) 104444 
2


Sources:
- 1-s2.0-S1566253526003246-main, page 1
- 1-s2.0-S1566253526003246-main, page 24
